In [23]:
# =============================================================================
# ANÁLISIS DE SENTIMIENTOS CON IMDB DATASET
# =============================================================================
# Temas cubiertos:
# 1. Regular Expressions (Limpieza de texto)
# 2. Text Summarization (Resumización)
# 3. Transformer Models (Análisis de sentimiento)
# =============================================================================

## 1. Preparación del Entorno
Instalamos e importamos las librerías necesarias para el análisis.

In [24]:
# Instalación de dependencias (ejecutar solo si es necesario)
# !pip install transformers torch pandas tqdm

In [25]:
# Importación de librerías
import pandas as pd
import re
import warnings
from tqdm import tqdm

warnings.filterwarnings('ignore')

print("Librerías básicas importadas correctamente")

Librerías básicas importadas correctamente


In [26]:
# Importación de modelos de Transformers
from transformers import pipeline

# Pipeline para resumización de texto
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# Pipeline para análisis de sentimiento
sentiment_analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

print("Modelos de Transformers cargados correctamente")

Device set to use cpu
Device set to use cpu


Modelos de Transformers cargados correctamente


## 2. Preparación del Dataset (Muestra)
Cargamos el dataset de IMDB y seleccionamos una muestra representativa para trabajar.

In [27]:
# Cargar el dataset de IMDB
df = pd.read_csv("IMDB Dataset.csv")

print(f"Dataset cargado:")
print(f"   - Total de reseñas: {len(df)}")
print(f"   - Columnas: {list(df.columns)}")
print(f"\nDistribución de sentimientos:")
print(df['sentiment'].value_counts())

Dataset cargado:
   - Total de reseñas: 50000
   - Columnas: ['review', 'sentiment']

Distribución de sentimientos:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [28]:
# Seleccionar una muestra balanceada para el ejercicio
# (5 positivas y 5 negativas para demostración)
muestra_positiva = df[df['sentiment'] == 'positive'].sample(5, random_state=42)
muestra_negativa = df[df['sentiment'] == 'negative'].sample(5, random_state=42)

# Combinar las muestras
df_muestra = pd.concat([muestra_positiva, muestra_negativa]).reset_index(drop=True)

print(f"Muestra seleccionada: {len(df_muestra)} reseñas")
print(f"   - Positivas: {len(df_muestra[df_muestra['sentiment'] == 'positive'])}")
print(f"   - Negativas: {len(df_muestra[df_muestra['sentiment'] == 'negative'])}")
print("\nVista previa de la primera reseña:")
print(df_muestra['review'].iloc[0][:300] + "...")

Muestra seleccionada: 10 reseñas
   - Positivas: 5
   - Negativas: 5

Vista previa de la primera reseña:
I don't know how or why this film has a meager rating on IMDb. This film, accompanied by "I am Curious: Blue" is a masterwork.<br /><br />The only thing that will let you down in this film is if you don't like the process of film, don't like psychology or if you were expecting hardcore pornographic ...


## 3. Limpieza de Texto con Regular Expressions (Regex)

Las **expresiones regulares** son patrones que nos permiten buscar y manipular texto de forma eficiente. 

### Patrones utilizados:
- `<br />` → Etiquetas HTML de salto de línea
- `[^a-zA-Z\s]` → Caracteres que NO son letras o espacios
- `\s+` → Múltiples espacios consecutivos

In [29]:
def limpiar_texto_regex(texto):
    """
    Función para limpiar texto usando expresiones regulares.
    
    Parámetros:
    - texto: string con el texto a limpiar
    
    Retorna:
    - texto limpio y normalizado
    """
    # 1. Eliminar etiquetas HTML (como <br />)
    texto = re.sub(r'<[^>]+>', ' ', texto)
    
    # 2. Eliminar caracteres especiales y números (mantener solo letras y espacios)
    texto = re.sub(r'[^a-zA-Z\s]', '', texto)
    
    # 3. Convertir múltiples espacios en uno solo
    texto = re.sub(r'\s+', ' ', texto)
    
    # 4. Eliminar espacios al inicio y final
    texto = texto.strip()
    
    # 5. Convertir a minúsculas
    texto = texto.lower()
    
    return texto

# Demostración de la limpieza
print("=" * 60)
print("DEMOSTRACIÓN DE LIMPIEZA CON REGEX")
print("=" * 60)

texto_ejemplo = df_muestra['review'].iloc[0][:500]
texto_limpio = limpiar_texto_regex(texto_ejemplo)

print("\nTEXTO ORIGINAL (primeros 500 caracteres):")
print("-" * 40)
print(texto_ejemplo)

print("\nTEXTO LIMPIO:")
print("-" * 40)
print(texto_limpio)

DEMOSTRACIÓN DE LIMPIEZA CON REGEX

TEXTO ORIGINAL (primeros 500 caracteres):
----------------------------------------
I don't know how or why this film has a meager rating on IMDb. This film, accompanied by "I am Curious: Blue" is a masterwork.<br /><br />The only thing that will let you down in this film is if you don't like the process of film, don't like psychology or if you were expecting hardcore pornographic ramming.<br /><br />This isn't a film that you will want to watch to unwind; it's a film that you want to see like any other masterpiece, with time, attention and care.<br /><br />******SUMMARIES, MAY

TEXTO LIMPIO:
----------------------------------------
i dont know how or why this film has a meager rating on imdb this film accompanied by i am curious blue is a masterwork the only thing that will let you down in this film is if you dont like the process of film dont like psychology or if you were expecting hardcore pornographic ramming this isnt a film that you will want t

In [30]:
# Aplicar limpieza a toda la muestra
df_muestra['review_limpio'] = df_muestra['review'].apply(limpiar_texto_regex)

print("Limpieza aplicada a todas las reseñas")
print("\nComparación de longitudes:")
print("-" * 40)

for i in range(3):
    original_len = len(df_muestra['review'].iloc[i])
    limpio_len = len(df_muestra['review_limpio'].iloc[i])
    reduccion = ((original_len - limpio_len) / original_len) * 100
    print(f"Reseña {i+1}: {original_len} → {limpio_len} caracteres ({reduccion:.1f}% reducción)")

Limpieza aplicada a todas las reseñas

Comparación de longitudes:
----------------------------------------
Reseña 1: 2612 → 2305 caracteres (11.8% reducción)
Reseña 2: 743 → 706 caracteres (5.0% reducción)
Reseña 3: 5411 → 5208 caracteres (3.8% reducción)


## 4. Text Summarization (Resumización)

La **resumización de texto** utiliza modelos de lenguaje para generar resúmenes concisos de textos largos.

### Modelo utilizado:
- **BART** (facebook/bart-large-cnn): Modelo pre-entrenado en CNN/DailyMail para generar resúmenes abstractivos.

In [31]:
def resumir_texto(texto, max_length=130, min_length=30):
    """
    Función para generar un resumen del texto usando BART.
    
    Parámetros:
    - texto: string con el texto a resumir
    - max_length: longitud máxima del resumen
    - min_length: longitud mínima del resumen
    
    Retorna:
    - resumen generado
    """
    try:
        # Limitar el texto de entrada (BART tiene límite de tokens)
        texto_truncado = texto[:1024]
        
        # Generar resumen
        resultado = summarizer(texto_truncado, 
                              max_length=max_length, 
                              min_length=min_length, 
                              do_sample=False)
        
        return resultado[0]['summary_text']
    except Exception as e:
        return f"Error al resumir: {str(e)}"

# Demostración de resumización
print("=" * 60)
print("DEMOSTRACIÓN DE TEXT SUMMARIZATION")
print("=" * 60)

texto_original = df_muestra['review_limpio'].iloc[0]
resumen = resumir_texto(texto_original)

print("\nTEXTO ORIGINAL (limpio):")
print("-" * 40)
print(texto_original[:600] + "...")
print(f"\nLongitud: {len(texto_original)} caracteres")

print("\nRESUMEN GENERADO:")
print("-" * 40)
print(resumen)
print(f"\nLongitud: {len(resumen)} caracteres")

DEMOSTRACIÓN DE TEXT SUMMARIZATION



TEXTO ORIGINAL (limpio):
----------------------------------------
i dont know how or why this film has a meager rating on imdb this film accompanied by i am curious blue is a masterwork the only thing that will let you down in this film is if you dont like the process of film dont like psychology or if you were expecting hardcore pornographic ramming this isnt a film that you will want to watch to unwind its a film that you want to see like any other masterpiece with time attention and care summaries may contain a spoiler or two the main thing about this film is that it blends the whole film within a film thing but it does it in such a way that sometimes you...

Longitud: 2305 caracteres

RESUMEN GENERADO:
----------------------------------------
i am curious blue is a masterwork the only thing that will let you down in this film is if you dont like the process of film dont like psychology or if you were expecting hardcore pornographic ramming. The film is like many films in one a pol

In [32]:
# Aplicar resumización a toda la muestra
print("Generando resúmenes para todas las reseñas...")
print("-" * 40)

resumenes = []
for i, texto in enumerate(tqdm(df_muestra['review_limpio'], desc="Resumiendo")):
    resumen = resumir_texto(texto)
    resumenes.append(resumen)

df_muestra['resumen'] = resumenes

print("\nResumización completada")
print("\nEjemplo de resúmenes generados:")
for i in range(3):
    print(f"\n--- Reseña {i+1} ({df_muestra['sentiment'].iloc[i]}) ---")
    print(f"Resumen: {df_muestra['resumen'].iloc[i]}")

Generando resúmenes para todas las reseñas...
----------------------------------------


Resumiendo: 100%|██████████| 10/10 [01:18<00:00,  7.81s/it]


Resumización completada

Ejemplo de resúmenes generados:

--- Reseña 1 (positive) ---
Resumen: i am curious blue is a masterwork the only thing that will let you down in this film is if you dont like the process of film dont like psychology or if you were expecting hardcore pornographic ramming. The film is like many films in one a political documentary about the social system in sweden at the time which in a lot of ways are still relevant to today.

--- Reseña 2 (positive) ---
Resumen: Despite the outlandish plots that are typical of farces the actors seemed to be trying to put something into their characters. When the extras from the music video attacked the evicting police you almost believed it was possible. The sex farce is also loaded with some very good nudity.

--- Reseña 3 (positive) ---
Resumen: One man james cole played by bruce willis in a heartwarming performance travels several decades to the past to retrieve information about a virus thats wiped out mankind and left onl

## 5. Análisis de Sentimiento con Transformers

El **análisis de sentimiento** clasifica el texto según la emoción o actitud expresada.

### Modelo utilizado:
- **DistilBERT** (distilbert-base-uncased-finetuned-sst-2-english): Versión optimizada de BERT, entrenada para clasificación de sentimientos (positivo/negativo).

In [33]:
def analizar_sentimiento(texto):
    """
    Función para analizar el sentimiento de un texto usando DistilBERT.
    
    Parámetros:
    - texto: string con el texto a analizar
    
    Retorna:
    - diccionario con etiqueta y puntuación de confianza
    """
    try:
        # Truncar texto para el modelo (límite de 512 tokens)
        texto_truncado = texto[:512]
        
        # Analizar sentimiento
        resultado = sentiment_analyzer(texto_truncado)
        
        return {
            'label': resultado[0]['label'],
            'score': resultado[0]['score']
        }
    except Exception as e:
        return {'label': 'ERROR', 'score': 0.0}

# Demostración de análisis de sentimiento
print("=" * 60)
print("DEMOSTRACIÓN DE ANÁLISIS DE SENTIMIENTO")
print("=" * 60)

# Analizar el resumen generado
texto_demo = df_muestra['resumen'].iloc[0]
resultado = analizar_sentimiento(texto_demo)

print(f"\nTexto analizado:")
print(f"   \"{texto_demo}\"")
print(f"\nResultado del análisis:")
print(f"   - Sentimiento: {resultado['label']}")
print(f"   - Confianza: {resultado['score']:.2%}")

DEMOSTRACIÓN DE ANÁLISIS DE SENTIMIENTO

Texto analizado:
   "i am curious blue is a masterwork the only thing that will let you down in this film is if you dont like the process of film dont like psychology or if you were expecting hardcore pornographic ramming. The film is like many films in one a political documentary about the social system in sweden at the time which in a lot of ways are still relevant to today."

Resultado del análisis:
   - Sentimiento: NEGATIVE
   - Confianza: 91.51%


In [34]:
# Aplicar análisis de sentimiento a todos los resúmenes
print("Analizando sentimientos de todas las reseñas...")
print("-" * 40)

sentimientos_predichos = []
confianzas = []

for resumen in tqdm(df_muestra['resumen'], desc="Analizando"):
    resultado = analizar_sentimiento(resumen)
    sentimientos_predichos.append(resultado['label'])
    confianzas.append(resultado['score'])

df_muestra['sentimiento_predicho'] = sentimientos_predichos
df_muestra['confianza'] = confianzas

print("\nAnálisis de sentimiento completado")

Analizando sentimientos de todas las reseñas...
----------------------------------------


Analizando: 100%|██████████| 10/10 [00:00<00:00, 31.04it/s]


Análisis de sentimiento completado


## 6. Pipeline Completo y Resultados Finales

Ahora ensamblamos todo el proceso en un pipeline completo y evaluamos los resultados.

In [35]:
def pipeline_completo(texto):
    """
    Pipeline completo de análisis de sentimientos.
    
    Pasos:
    1. Limpieza con Regex
    2. Resumización con BART
    3. Análisis de sentimiento con DistilBERT
    
    Parámetros:
    - texto: texto original de la reseña
    
    Retorna:
    - diccionario con todos los resultados
    """
    # Paso 1: Limpieza con Regex
    texto_limpio = limpiar_texto_regex(texto)
    
    # Paso 2: Resumización
    resumen = resumir_texto(texto_limpio)
    
    # Paso 3: Análisis de sentimiento
    sentimiento = analizar_sentimiento(resumen)
    
    return {
        'texto_original': texto[:200] + '...',
        'texto_limpio': texto_limpio[:200] + '...',
        'resumen': resumen,
        'sentimiento': sentimiento['label'],
        'confianza': sentimiento['score']
    }

# Demostración del pipeline completo
print("=" * 70)
print("DEMOSTRACIÓN DEL PIPELINE COMPLETO")
print("=" * 70)

# Procesar una reseña de ejemplo
resultado = pipeline_completo(df_muestra['review'].iloc[0])

print("\nENTRADA:")
print(f"   {resultado['texto_original']}")

print("\nPASO 1 - Limpieza (Regex):")
print(f"   {resultado['texto_limpio']}")

print("\nPASO 2 - Resumización (BART):")
print(f"   {resultado['resumen']}")

print("\nPASO 3 - Análisis de Sentimiento (DistilBERT):")
print(f"   Sentimiento: {resultado['sentimiento']}")
print(f"   Confianza: {resultado['confianza']:.2%}")

DEMOSTRACIÓN DEL PIPELINE COMPLETO

ENTRADA:
   I don't know how or why this film has a meager rating on IMDb. This film, accompanied by "I am Curious: Blue" is a masterwork.<br /><br />The only thing that will let you down in this film is if you d...

PASO 1 - Limpieza (Regex):
   i dont know how or why this film has a meager rating on imdb this film accompanied by i am curious blue is a masterwork the only thing that will let you down in this film is if you dont like the proce...

PASO 2 - Resumización (BART):
   i am curious blue is a masterwork the only thing that will let you down in this film is if you dont like the process of film dont like psychology or if you were expecting hardcore pornographic ramming. The film is like many films in one a political documentary about the social system in sweden at the time which in a lot of ways are still relevant to today.

PASO 3 - Análisis de Sentimiento (DistilBERT):
   Sentimiento: NEGATIVE
   Confianza: 91.51%


In [36]:
# Tabla de resultados finales
print("=" * 70)
print("TABLA DE RESULTADOS FINALES")
print("=" * 70)

# Mapear etiquetas para comparación
df_muestra['sentiment_real_mapped'] = df_muestra['sentiment'].map({
    'positive': 'POSITIVE', 
    'negative': 'NEGATIVE'
})

# Verificar aciertos
df_muestra['acierto'] = df_muestra['sentiment_real_mapped'] == df_muestra['sentimiento_predicho']

# Mostrar tabla de resultados
resultados_tabla = df_muestra[['sentiment', 'sentimiento_predicho', 'confianza', 'acierto']].copy()
resultados_tabla.columns = ['Real', 'Predicho', 'Confianza', 'Acierto']
resultados_tabla['Confianza'] = resultados_tabla['Confianza'].apply(lambda x: f"{x:.2%}")
resultados_tabla['Acierto'] = resultados_tabla['Acierto'].apply(lambda x: 'SI' if x else 'NO')

print("\n")
print(resultados_tabla.to_string(index=True))

# Calcular accuracy
accuracy = df_muestra['acierto'].mean() * 100
print(f"\nPRECISIÓN DEL MODELO: {accuracy:.1f}%")
print(f"   Aciertos: {df_muestra['acierto'].sum()} / {len(df_muestra)}")

TABLA DE RESULTADOS FINALES


       Real  Predicho Confianza Acierto
0  positive  NEGATIVE    91.51%      NO
1  positive  POSITIVE    99.57%      SI
2  positive  POSITIVE    99.74%      SI
3  positive  NEGATIVE    99.87%      NO
4  positive  NEGATIVE    99.93%      NO
5  negative  NEGATIVE    94.79%      SI
6  negative  NEGATIVE    99.74%      SI
7  negative  POSITIVE    91.55%      NO
8  negative  NEGATIVE    99.05%      SI
9  negative  NEGATIVE    98.91%      SI

PRECISIÓN DEL MODELO: 60.0%
   Aciertos: 6 / 10


In [37]:
# Mostrar ejemplos detallados
print("=" * 70)
print("EJEMPLOS DETALLADOS")
print("=" * 70)

for i in range(min(3, len(df_muestra))):
    print(f"\n{'─' * 70}")
    print(f"RESEÑA #{i+1}")
    print(f"{'─' * 70}")
    print(f"\nSentimiento Real: {df_muestra['sentiment'].iloc[i].upper()}")
    print(f"Sentimiento Predicho: {df_muestra['sentimiento_predicho'].iloc[i]}")
    print(f"Confianza: {df_muestra['confianza'].iloc[i]:.2%}")
    print(f"\nResumen generado:")
    print(f"   \"{df_muestra['resumen'].iloc[i]}\"")
    print(f"\n{'CORRECTO' if df_muestra['acierto'].iloc[i] else 'INCORRECTO'}")

EJEMPLOS DETALLADOS

──────────────────────────────────────────────────────────────────────
RESEÑA #1
──────────────────────────────────────────────────────────────────────

Sentimiento Real: POSITIVE
Sentimiento Predicho: NEGATIVE
Confianza: 91.51%

Resumen generado:
   "i am curious blue is a masterwork the only thing that will let you down in this film is if you dont like the process of film dont like psychology or if you were expecting hardcore pornographic ramming. The film is like many films in one a political documentary about the social system in sweden at the time which in a lot of ways are still relevant to today."

INCORRECTO

──────────────────────────────────────────────────────────────────────
RESEÑA #2
──────────────────────────────────────────────────────────────────────

Sentimiento Real: POSITIVE
Sentimiento Predicho: POSITIVE
Confianza: 99.57%

Resumen generado:
   "Despite the outlandish plots that are typical of farces the actors seemed to be trying to put somethin

## Resumen del Pipeline

| Etapa | Técnica | Herramienta | Descripción |
|-------|---------|-------------|-------------|
| **1. Limpieza** | Regular Expressions | `re` (Python) | Elimina HTML, caracteres especiales y normaliza espacios |
| **2. Resumización** | Text Summarization | `BART` (Hugging Face) | Genera resúmenes abstractivos del texto |
| **3. Clasificación** | Transformer Models | `DistilBERT` (Hugging Face) | Clasifica el sentimiento como positivo/negativo |

### Ventajas del Pipeline:
- Automatización completa del análisis
- Reducción significativa del texto a procesar
- Alta precisión gracias a modelos pre-entrenados
- Escalable a grandes volúmenes de datos

In [38]:
# Guardar resultados en CSV
df_resultados = df_muestra[['review', 'sentiment', 'review_limpio', 'resumen', 'sentimiento_predicho', 'confianza', 'acierto']]
df_resultados.to_csv('resultados_analisis_sentimientos.csv', index=False)

print("Resultados guardados en 'resultados_analisis_sentimientos.csv'")
print("\n¡Pipeline completado exitosamente!")

Resultados guardados en 'resultados_analisis_sentimientos.csv'

¡Pipeline completado exitosamente!
